# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**My lane: Lane 2 — Refresh / Content Opportunity Scoring.**

I'm choosing this lane over the others because it produces a concrete, usable output — a ranked review queue — rather than only a report. It also lines up directly with the starter pipeline I already ran end to end (`scripts/01`–`05`): I've seen the baseline rule score, the trained model, and the client-holdout evaluation, so I already know this data supports a real precision@K lift over a hand-written rule (random forest hit precision@50 = 0.740 vs. the baseline rule's 0.240 in the starter run). That gap is the whole argument for doing this instead of only writing an if-statement: the pattern is real, but it's tangled enough across many signals that a plain rule leaves a lot of value on the table. I also have prior experience building this exact shape of project — a scored/ranked classification pipeline with an explainable baseline (I did this for a churn-prediction project and an energy-consumption project in other internships) — so I can move fast here and spend the saved time on getting the framing and validation right instead of relearning the pipeline shape from scratch.


In [1]:
# Nothing to run for this section — this is a framing choice, not a data check.
print("Lane: Refresh / Content Opportunity Scoring (Lane 2)")


Lane: Refresh / Content Opportunity Scoring (Lane 2)


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Unit of analysis:** one row = one content item (a page) for one client, summarized over a trailing 90-day window. `content_id` is the grain; `client_id` groups pages that belong to the same account.

**The decision:** out of a client's whole content inventory, *which pages should a content reviewer look at first this week*, given that review capacity is small and fixed (a reviewer can realistically check maybe 20–50 pages, not thousands).

**Who acts, and what they do:** a FlyRank content strategist / account manager, working through a ranked queue. For a page near the top of the queue, they open it, read the reason code(s) attached to it, and decide on an action — refresh the content, expand thin sections, protect a page that's still doing fine, or leave it alone and move to the next candidate. The output only has a customer if it changes what they'd have looked at first; a queue that just re-sorts pages randomly has no one acting differently because of it.

**Cost of a wrong call:**
- **False positive** (a page is ranked high but isn't actually a real problem): the reviewer spends limited hours on a page that didn't need it, which is a real opportunity cost — that hour didn't go to a page that *did* need attention.
- **False negative** (a page that's quietly losing visibility never makes the queue): the client keeps losing impressions/clicks on a page nobody ever reviews, which is a slower, silent version of the same lost-revenue problem, and harder to catch after the fact.
Because review time is the scarce resource, precision near the top of the ranking (precision@K) matters more here than perfect accuracy across every page.

**Why data or ML helps at all:** with 30,000 content items across 32 clients in just the starter slice (519,606 items across 104 clients in the full warehouse), and — as I show below — more than half of the starter slice currently trending down, nobody can manually review "everything that looks declining." A simple hand-written rule (the starter baseline) already helps, but the starter pipeline's own evaluation shows a trained model roughly triples precision@50 versus that rule. That's the signal that this problem is "real pattern, too tangled to hand-code" rather than "just write an if-statement," which is exactly the bar this lane is supposed to clear.


In [2]:
# Nothing to run for this section either — framing again, not a data check.
print("Decision: which pages to review first, given limited reviewer capacity")
print("Action: refresh / expand / protect / prune / monitor, chosen by a human reviewer from a ranked queue")
print("Cost of a wrong call: wasted reviewer hours (false positive) or a silently worsening page (false negative)")


Decision: which pages to review first, given limited reviewer capacity
Action: refresh / expand / protect / prune / monitor, chosen by a human reviewer from a ranked queue
Cost of a wrong call: wasted reviewer hours (false positive) or a silently worsening page (false negative)


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*


In [3]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# same filter the starter pipeline uses, so these numbers match what the pipeline actually sees
f = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id")

n_rows = len(f)
n_clients = f["client_id"].nunique()
pct_down = (f["trend_direction"] == "down").mean() * 100
declining_with_demand = ((f["trend_direction"] == "down") & (f["impressions_90d"] >= 100)).sum()
low_ctr_visible = ((f["impressions_90d"] >= 500) & (f["avg_position"] > 0) & (f["avg_position"] <= 20) & (f["ctr"] < 0.5)).sum()

print(f"Rows after the starter filter: {n_rows:,} content items across {n_clients} clients")
print(f"Share of pages currently trending down: {pct_down:.1f}%")
print(f"Pages flagged 'declining_with_demand' (down trend + impressions_90d >= 100): {declining_with_demand:,}")
print(f"Pages flagged 'low_ctr_visible_page' (decent position, weak CTR, real volume): {low_ctr_visible:,}")


Rows after the starter filter: 30,000 content items across 32 clients
Share of pages currently trending down: 54.2%
Pages flagged 'declining_with_demand' (down trend + impressions_90d >= 100): 13,152
Pages flagged 'low_ctr_visible_page' (decent position, weak CTR, real volume): 9,759


**What these numbers say:** in just the 30k-row starter slice, over half the pages (54.2%) are currently trending down, and 13,152 pages (43.8% of the slice) already clear the bar for "declining with real demand behind it." On top of that, 9,759 pages (32.5%) are ranked well enough to matter but are under-capturing clicks for their position. No content team reviews thirteen thousand pages by hand — a ranked queue is the only realistic way anyone acts on this, which is the whole case for this lane over the next 7 weeks.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:**
- **Observed / decision-support results.** My output is a ranked list of pages worth a human looking at first, built from observed search and engagement signals measured over a defined window. I can say a page *shows evidence* of decline, weak CTR for its position, or thinning demand.
- **Directional, evidence-backed comparisons.** I can compare my ranking against the transparent baseline rule and report precision@K, recall, and where the two disagree — a fair "did the extra complexity earn its keep" comparison.
- **A defined proxy, named as a proxy.** The starter label (`trend_direction == "down"`) is a bucket computed from the *current* window, not a future outcome — I'll say that plainly rather than treat it as ground truth. My provisional plan for later weeks is to move to a real forward-looking label (features from a prior window predicting an outcome in a later window) once I get to the data-contract and leakage stages.

**What I will never claim:**
- That refreshing a page **caused** a recovery — I have no experiment or causal design here, only observational data. At most I can say a page *was refreshed and later recovered*, never that the refresh is why.
- That I've reverse-engineered or "predicted Google's algorithm." Position, CTR, and impressions are observed outcomes, not the ranking factors behind them.
- That a high score is a guarantee — it is a prioritization signal for a limited-capacity human reviewer, not a verdict on any single page.
- Anything about a specific client, URL, query, or page identity — the dataset is pseudonymized and I'll keep every example and write-up that way.


In [4]:
# Self-documentation only, nothing to compute here.
print("Claims allowed: observed, directional, decision-support")
print("Claims NOT allowed: causal ('refresh caused recovery'), algorithm-prediction, guaranteed outcomes")


Claims allowed: observed, directional, decision-support
Claims NOT allowed: causal ('refresh caused recovery'), algorithm-prediction, guaranteed outcomes


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
